In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

In [81]:
def load_fred(path, col_name):
    df = pd.read_csv(path, parse_dates=['observation_date'])
    df.columns = ['date', col_name]
    df[col_name] = pd.to_numeric(df[col_name], errors='coerce')
    df = df.set_index('date')
    df = df.dropna()
    return df

In [ ]:
houst1f   = load_fred('HOUST1F.csv',        'housing_starts') #Housing units supply
msacsr    = load_fred('MSACSR.csv',          'months_of_supply')
actlis    = load_fred('ACTLISCOUUS.csv',     'active_listings')   #Availble for sale
csushpisa = load_fred('CSUSHPISA.csv',       'home_price_index')
mspus     = load_fred('MSPUS.csv',           'median_home_price')
mortgage  = load_fred('MORTGAGE30US.csv',    'mortgage_rate')

In [83]:
print(houst1f.head())
print(msacsr.head())
print(actlis.head())
print(csushpisa.head())
print(mspus.head())
print(mortgage.head())

            housing_starts
date                      
2000-01-01            1268
2000-02-01            1255
2000-03-01            1313
2000-04-01            1275
2000-05-01            1230
            months_of_supply
date                        
2000-01-01               4.3
2000-02-01               4.3
2000-03-01               4.3
2000-04-01               4.4
2000-05-01               4.4
            active_listings
date                       
2016-07-01          1463025
2016-08-01          1460071
2016-09-01          1443103
2016-10-01          1407722
2016-11-01          1339724
            home_price_index
date                        
2000-01-01           100.551
2000-02-01           101.339
2000-03-01           102.126
2000-04-01           102.922
2000-05-01           103.677
            median_home_price
date                         
2000-01-01             165300
2000-04-01             163200
2000-07-01             168800
2000-10-01             172900
2001-01-01             169800

In [84]:
# Turn weekly to monthly value
mortgage = mortgage.resample('MS').mean()
print(mortgage)
#Turn quarter to monthly value (forward fill)
mspus = mspus.resample('MS').ffill()
print(mspus)

            mortgage_rate
date                     
2000-01-01         8.2100
2000-02-01         8.3250
2000-03-01         8.2400
2000-04-01         8.1525
2000-05-01         8.5150
...                   ...
2025-12-01         6.1900
2026-01-01         6.1025
2026-02-01         6.0475
2026-03-01         6.1775
2026-04-01         6.3320

[316 rows x 1 columns]
            median_home_price
date                         
2000-01-01             165300
2000-02-01             165300
2000-03-01             165300
2000-04-01             163200
2000-05-01             163200
...                       ...
2025-06-01             416100
2025-07-01             410100
2025-08-01             410100
2025-09-01             410100
2025-10-01             405300

[310 rows x 1 columns]


In [85]:
# Merge ALL together...
merge = houst1f.merge(msacsr, how = 'outer', on = 'date')
merge = merge.merge(actlis, how = 'outer', on = 'date')
merge = merge.merge(csushpisa, how = 'outer', on = 'date')
merge = merge.merge(mspus, how = 'outer', on = 'date')
merge = merge.merge(mortgage, how = 'outer', on = 'date')
print(merge)

            housing_starts  months_of_supply  active_listings  \
date                                                            
2000-01-01          1268.0               4.3              NaN   
2000-02-01          1255.0               4.3              NaN   
2000-03-01          1313.0               4.3              NaN   
2000-04-01          1275.0               4.4              NaN   
2000-05-01          1230.0               4.4              NaN   
...                    ...               ...              ...   
2025-12-01           950.0               8.0         976833.0   
2026-01-01           898.0               9.7         912696.0   
2026-02-01           941.0               NaN         915717.0   
2026-03-01          1032.0               NaN         947866.0   
2026-04-01             NaN               NaN        1002935.0   

            home_price_index  median_home_price  mortgage_rate  
date                                                            
2000-01-01           100

In [ ]:
merge = merge[merge.index <=  '2025-10-01' ] #Keep from 2000 to 10/2025 only
print(merge)

            housing_starts  months_of_supply  active_listings  \
date                                                            
2000-01-01          1268.0               4.3              NaN   
2000-02-01          1255.0               4.3              NaN   
2000-03-01          1313.0               4.3              NaN   
2000-04-01          1275.0               4.4              NaN   
2000-05-01          1230.0               4.4              NaN   
...                    ...               ...              ...   
2025-06-01           925.0               9.1        1082520.0   
2025-07-01           951.0               9.3        1102787.0   
2025-08-01           869.0               8.4        1098681.0   
2025-09-01           836.0               8.1        1100407.0   
2025-10-01           894.0               9.0        1099856.0   

            home_price_index  median_home_price  mortgage_rate  
date                                                            
2000-01-01           100

In [ ]:
merge.to_csv("housing_dashboard_data.csv") #Export to csv file

In [ ]:
print("\n" + "="*60)
print("CLAIM 1: Housing starts collapsed and never recovered")
print("="*60)
 
pre_crisis_peak  = houst1f.loc['2000':'2006','housing_starts'].max()
crisis_trough    = houst1f.loc['2008':'2012','housing_starts'].min()
recent_avg       = houst1f.loc['2020':'2024','housing_starts'].mean()
pre_crisis_avg   = houst1f.loc['2000':'2006','housing_starts'].mean()
pct_drop         = (crisis_trough - pre_crisis_peak) / pre_crisis_peak * 100
pct_recovery     = (recent_avg - pre_crisis_avg) / pre_crisis_avg * 100
 
print(f"  Pre-crisis peak (2000-2006):       {pre_crisis_peak:,.0f}k units/yr")
print(f"  Crisis trough (2008-2012):         {crisis_trough:,.0f}k units/yr")
print(f"  Drop from peak to trough:          {pct_drop:.1f}%")
print(f"  Pre-crisis avg (2000-2006):        {pre_crisis_avg:,.0f}k units/yr")
print(f"  Recent avg (2020-2024):            {recent_avg:,.0f}k units/yr")
print(f"  Recovery vs pre-crisis avg:        {pct_recovery:.1f}%")
print(f"  VERDICT: Still {abs(pct_recovery):.0f}% below pre-crisis average" if pct_recovery < 0
      else f"  VERDICT: Fully recovered (+{pct_recovery:.0f}%)")


CLAIM 1: Housing starts collapsed and never recovered
  Pre-crisis peak (2000-2006):       1,823k units/yr
  Crisis trough (2008-2012):         353k units/yr
  Drop from peak to trough:          -80.6%
  Pre-crisis avg (2000-2006):        1,453k units/yr
  Recent avg (2020-2024):            1,019k units/yr
  Recovery vs pre-crisis avg:        -29.8%
  VERDICT: Still 30% below pre-crisis average


In [ ]:
#Load raw file
df = pd.read_excel("histtab12.xlsx", header = None)
df

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14
0,Table with row headings in Column A and column...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Table 12. Household Estimates for the United S...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,(Numbers in thousands),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Age of Householder,19821,NaN,1983,NaN,1984,NaN,1985,NaN,1986,NaN,1987,NaN,1988,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
177,1 The age categories for 1982 through 1988 do ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
178,"2 Beginning in 1989, the not reported cases we...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
179,"*Due to a lapse in federal funding, the Curren...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
180,r1 Revised based on the 1990 Census.,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [108]:
#Each block header row start at these position
block_starts = [4,30,54,78,102,126,150]

In [122]:
#The 5 group ages band we want
age_labels = ['Under 35', '35 to 44', '45 to 54', '55 to 64', '65 and over']

In [136]:
#Extract data
results = []

for bs in block_starts:
  ##Get the years from the header row
  # Years are in every column starting from 1
  year_row = df.iloc[bs, 1:]
  #print(year_row)
  years = []
  for i in range(0, len(year_row),2):
    val = year_row.iloc[i]
    #print(val)
    
    if pd.notna(val):
      # Clean year - remove suffixes like r1, r2
      yr = str(val)[:4]
      #print(yr)
      years.append((i+1, yr))   #(column index, year)
  #print(years)   ##tells where to look in to find value for Total and Owner of specific year

  #Get the grouped age band rows
  for age_offset, age_label in zip(range(18,23), age_labels):
    row_idx = bs + age_offset
    row = df.iloc[row_idx]   #Get entire rows of value of years of the block
    #print(row)

    #For each year, grab Total and Owner values
    for col_idx, year in years:
      if col_idx +1 < len(df.columns):
        total = pd.to_numeric(row.iloc[col_idx])
        owner = pd.to_numeric(row.iloc[col_idx+1])

        #Calculate onwership rate
        if pd.notna(total) and pd.notna(owner) and total >0:
          rate = owner /total *100
          results.append({
            'year': int(year),
            'age_group': age_label,
            'total': round(total,1),
            'owner': round(owner,1),
            'ownership_rate': round(rate, 1)
          })


  



In [150]:
result_df = (
    pd.DataFrame(results)
    .drop_duplicates(subset=['year', 'age_group'])
    .sort_values(['year', "age_group"])
    .reset_index(drop=True)
)

In [153]:
result_df

,year,age_group,total,owner,ownership_rate
0,1982,35 to 44,15298.0,10710.9,70.0
1,1982,45 to 54,12540.0,9708.7,77.4
2,1982,55 to 64,12957.0,10369.6,80.0
3,1982,65 and over,17440.0,12978.2,74.4
4,1982,Under 35,24860.0,10240.8,41.2
...,...,...,...,...,...
215,2025,35 to 44,23396.0,14228.0,60.8
216,2025,45 to 54,21511.0,15025.0,69.8
217,2025,55 to 64,22763.0,17267.0,75.9
218,2025,65 and over,38483.0,30244.0,78.6


In [155]:
result_df.to_csv("homeownership_by_age.csv", index = False)

In [4]:
# Load zillow file 
zillow_file = pd.read_csv(r"D:\Spring 2026\Data 201\Final Project\Metro_zhvi_uc_sfrcondo_tier_0.33_0.67_sm_sa_month.csv")
zillow_file.head()

,RegionID,SizeRank,RegionName,RegionType,StateName,2000-01-31,2000-02-29,2000-03-31,2000-04-30,2000-05-31,...,2025-06-30,2025-07-31,2025-08-31,2025-09-30,2025-10-31,2025-11-30,2025-12-31,2026-01-31,2026-02-28,2026-03-31
0,102001,0,United States,country,NaN,122723.314722,122939.224374,123206.984135,123781.276262,124442.217647,...,362560.842583,362132.841288,361904.476773,362146.820735,362582.459574,363260.484023,364053.711171,364810.501924,365526.782311,366018.815220
1,394913,1,"New York, NY",msa,NY,217784.554624,218710.016103,219644.049023,221536.761145,223496.730091,...,692740.899026,693911.257366,694699.788803,696097.775239,698428.981973,701765.079321,705178.795698,708236.709315,711803.048283,715583.951353
2,753899,2,"Los Angeles, CA",msa,CA,224762.443660,225598.845127,226712.923903,228929.566423,231353.878418,...,957170.045940,954540.625630,953453.887828,954525.850586,956976.849684,960342.016204,964358.917595,966925.310466,968174.896554,967836.201342
3,394463,3,"Chicago, IL",msa,IL,153387.535332,153529.527506,153799.954104,154472.708272,155281.950617,...,332166.351158,333005.360528,334000.754039,335333.297892,336637.606600,338184.663240,339916.392330,341535.828444,343281.802377,344686.992147
4,394514,4,"Dallas, TX",msa,TX,129543.865799,129601.565090,129667.896331,129840.151596,130067.265447,...,371756.987502,369710.155065,368327.677314,367670.920611,367317.108777,367049.499400,366828.525223,366388.507924,365804.694106,364734.092138


In [9]:
select_zillow = zillow_file[["RegionName", "RegionType", "StateName", "2020-12-31", "2026-03-31"]]
select_zillow

,RegionName,RegionType,StateName,2020-12-31,2026-03-31
0,United States,country,NaN,271027.769078,366018.815220
1,"New York, NY",msa,NY,528873.893503,715583.951353
2,"Los Angeles, CA",msa,CA,735279.017034,967836.201342
3,"Chicago, IL",msa,IL,252509.481601,344686.992147
4,"Dallas, TX",msa,TX,281917.110322,364734.092138
...,...,...,...,...,...
890,"Zapata, TX",msa,TX,162653.595969,131019.512465
891,"Ketchikan, AK",msa,AK,324916.010703,393126.319028
892,"Craig, CO",msa,CO,203733.679599,301863.226537
893,"Vernon, TX",msa,TX,110736.330950,99552.466297


In [11]:
select_zillow[["2020-12-31", "2026-03-31"]] = select_zillow[["2020-12-31", "2026-03-31"]].astype(float)

In [12]:
select_zillow.to_csv("clean_zillow.csv")